In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50
from tqdm.auto import tqdm

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
DATASET_ROOT = Path("/media/imagenesmedicas/DATA1/01-ImagenesMedicas-US1/02-Databases/Mammo-Bench/c86fb00c-0fb8-4e0e-85a2-4d415f9c1ada_1a9410d8-9769-4064-a064-0160f2fd193d_DATASET-FILE_Mammo_Bench_zip_20241225112148174/Mammo_Data/Mammo-Bench")

class CSVDataset(Dataset):
    def __init__(self, csv_file, split, root_dir="", transform=None, label_to_idx=None):
        self.data = pd.read_csv(csv_file)
        self.data = self.data[self.data["split"] == split].reset_index(drop=True)

        self.root_dir = Path(root_dir)
        self.transform = transform

        if label_to_idx is None:
            labels = sorted(self.data["classification"].unique())
            self.label_to_idx = {label: i for i, label in enumerate(labels)}
        else:
            self.label_to_idx = label_to_idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        img_path = Path(self.root_dir) / Path(row["preprocessed_image_path"])
        image = Image.open(img_path).convert("RGB")

        label = self.label_to_idx[row["classification"]]

        if self.transform:
            image = self.transform(image)

        return image, label

In [4]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5)),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

In [5]:
MANIFEST_PATH = "/media/imagenesmedicas/DATA1/01-ImagenesMedicas-US1/13-PregradoJulian/Federal Learning/infraestructura federada/Federal-Learning/manifests/mamo-bench-split-no-ddsm-rsna.csv"

train_processed = CSVDataset(csv_file=MANIFEST_PATH, split="train", root_dir=DATASET_ROOT, transform=train_transform)
# val/test reuse train's label_to_idx so "Benign"/"Malignant" map to the same
# class index in every split (previously each split computed its own mapping
# independently from its local sorted unique labels).
validation_processed = CSVDataset(csv_file=MANIFEST_PATH, split="val", root_dir=DATASET_ROOT, transform=val_transform, label_to_idx=train_processed.label_to_idx)
test_processed = CSVDataset(csv_file=MANIFEST_PATH, split="test", root_dir=DATASET_ROOT, transform=val_transform, label_to_idx=train_processed.label_to_idx)

In [6]:
train = DataLoader(train_processed, batch_size=32, shuffle=True)
validation = DataLoader(validation_processed, batch_size=32, shuffle=False)
test = DataLoader(test_processed, batch_size=32, shuffle=False)

In [7]:
class Classifier(nn.Module):
    def __init__(self, num_class):
        super().__init__()
        
        self.linear1    = nn.Linear(2048, 1024)
        self.bn1        = nn.BatchNorm1d(1024)
        self.act1       = nn.LeakyReLU(0.2)
        self.dropout1   = nn.Dropout(0.5)
        
        self.linear2 = nn.Linear(1024, 512)
        self.bn2        = nn.BatchNorm1d(512)
        self.act2       = nn.LeakyReLU(0.2)
        self.dropout2   = nn.Dropout(0.5)
        
        self.linear3 = nn.Linear(512, num_class)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.linear1(x)
        x = self.bn1(x)
        x = self.act1(x)
        x = self.dropout1(x)
        x = self.linear2(x)
        x = self.bn2(x)
        x = self.act2(x)
        x = self.dropout2(x)
        x = self.linear3(x)
        return x


class Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        base_model = resnet50(weights=None)
        encoder_layers = list(base_model.children())
        self.backbone = nn.Sequential(*encoder_layers[:9])
                        
    def forward(self, x):
        return self.backbone(x)

In [8]:
# backbone is created below from RadImageNet weights (load_radimagenet_backbone);
# only the classifier head needs its own fresh init here.
classifier = Classifier(num_class=2).to(device)

In [9]:
# Example: load RadImageNet weights into a ResNet50 backbone
# Replace this path with the actual .pth or .pt file you downloaded.
radimagenet_weights_path = "/media/imagenesmedicas/DATA1/01-ImagenesMedicas-US1/13-PregradoJulian/Federal Learning/infraestructura federada/Federal-Learning/weights/RadImageNet-resnet50.pth"


def load_radimagenet_backbone(weights_path, device="cpu"):
    model = resnet50(weights=None)
    checkpoint = torch.load(weights_path, map_location=device)

    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    else:
        state_dict = checkpoint

    # Remove common prefixes from checkpoints
    state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}

    # The official RadImageNet-resnet50.pth stores the backbone as
    # nn.Sequential([conv1, bn1, relu, maxpool, layer1..4, avgpool]), so keys
    # look like "backbone.0.weight" instead of "conv1.weight". Without this
    # remap, load_state_dict(strict=False) matches ZERO keys and silently
    # leaves the model at its random init (same bug class documented in
    # src/fedmammobench/models/weight_loaders/_keymaps.py, which this mirrors).
    backbone_remap = {
        "backbone.0.": "conv1.",
        "backbone.1.": "bn1.",
        "backbone.4.": "layer1.",
        "backbone.5.": "layer2.",
        "backbone.6.": "layer3.",
        "backbone.7.": "layer4.",
    }
    remapped_state_dict = {}
    for k, v in state_dict.items():
        new_k = k
        for old_prefix, new_prefix in backbone_remap.items():
            if k.startswith(old_prefix):
                new_k = new_prefix + k[len(old_prefix):]
                break
        remapped_state_dict[new_k] = v
    state_dict = remapped_state_dict

    # Keep only the ResNet50 backbone keys
    state_dict = {
        k: v for k, v in state_dict.items()
        if k.startswith(("conv1", "bn1", "relu", "maxpool", "layer1", "layer2", "layer3", "layer4", "avgpool"))
    }

    if len(state_dict) == 0:
        raise RuntimeError(
            "0 RadImageNet tensors matched the model after remapping — "
            "check the checkpoint's key format before continuing."
        )

    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(f"Loaded {len(state_dict)} RadImageNet tensors into the backbone.")
    print("Missing keys:", missing)
    print("Unexpected keys:", unexpected)

    # Truncate to conv1..avgpool, matching the Backbone class above, so its
    # output shape (batch, 2048, 1, 1) lines up with Classifier's Linear(2048, ...)
    encoder_layers = list(model.children())
    return nn.Sequential(*encoder_layers[:9])


backbone = load_radimagenet_backbone(radimagenet_weights_path, device=device).to(device)

Loaded 318 RadImageNet tensors into the backbone.
Missing keys: ['fc.weight', 'fc.bias']
Unexpected keys: []


In [10]:
# exp72: solo la última capa del backbone (layer4) queda descongelada;
# conv1/bn1/layer1/layer2/layer3 se congelan (requires_grad=False).
# backbone = Sequential(conv1[0], bn1[1], relu[2], maxpool[3], layer1[4],
# layer2[5], layer3[6], layer4[7], avgpool[8]) — mismo orden que
# backbone_remap en la celda anterior, así que backbone[7] es layer4.
for param in backbone.parameters():
    param.requires_grad = False
 

trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
total = sum(p.numel() for p in backbone.parameters())
print(f"Backbone trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

Backbone trainable params: 0 / 23,508,032 (0.00%)


In [ ]:
RUN_DIR = Path("/media/imagenesmedicas/DATA1/01-ImagenesMedicas-US1/13-PregradoJulian/Federal Learning/infraestructura federada/Federal-Learning/runs/exp74_resnet50_radimagenet_alllayerfreeze")
RUN_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = RUN_DIR / "best_model.pth"

# Per-epoch loss history, populated by train_model() below and consumed by
# the loss-curve cell right after `model = train_model(...)`. Module-level
# (not returned by train_model) so the existing `model = train_model(...)`
# call site doesn't need to change to a tuple unpack.
train_loss_history = []
val_loss_history = []


def freeze_bn_running_stats(module):
    # Capas BatchNorm con gamma/beta congelados se fuerzan a eval() para que
    # running_mean/running_var no sigan derivando en el forward pass, aunque
    # sus parámetros afines no reciban gradiente (mismo fix que Trainer._freeze_bn_running_stats
    # en el paquete fedmammobench principal).
    for m in module.modules():
        if isinstance(m, nn.BatchNorm2d) and not m.weight.requires_grad:
            m.eval()


def train_model(model, criterion, optimizer, num_epochs=50, sheduler=None):
    min_valid_loss = np.inf
    train_loss_history.clear()
    val_loss_history.clear()
    counter = 0
    patience = 10  

    epoch_bar = tqdm(range(num_epochs), desc="Epochs", unit="epoch")
    for e in epoch_bar:
        train_loss = 0.0
        model.train()
        freeze_bn_running_stats(model)
        train_bar = tqdm(train, desc=f"  train {e+1}/{num_epochs}", unit="batch", leave=False)
        for data, labels in train_bar:
            data, labels = data.to(device, dtype=torch.float), labels.to(device)

            optimizer.zero_grad()
            target = model(data)
            loss = criterion(target, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            train_bar.set_postfix(loss=f"{loss.item():.4f}")

        valid_loss = 0.0
        model.eval()
        val_bar = tqdm(validation, desc=f"  val   {e+1}/{num_epochs}", unit="batch", leave=False)
        with torch.no_grad():
            for data, labels in val_bar:
                data, labels = data.to(device, dtype=torch.float), labels.to(device)

                target = model(data)
                loss = criterion(target, labels)
                valid_loss += loss.item()
                val_bar.set_postfix(loss=f"{loss.item():.4f}")

        avg_train_loss = train_loss / len(train)
        avg_valid_loss = valid_loss / len(validation)
        train_loss_history.append(avg_train_loss)
        val_loss_history.append(avg_valid_loss)
        epoch_bar.set_postfix(train_loss=f"{avg_train_loss:.4f}", val_loss=f"{avg_valid_loss:.4f}")

        print(f'Epoch {e+1} \t\t Training Loss: {avg_train_loss} \t\t Validation Loss: {avg_valid_loss}')
        if min_valid_loss > valid_loss:
            print(f'Validation Loss Decreased({min_valid_loss:.6f}--->{valid_loss:.6f}) \t Saving The Model')
            min_valid_loss = valid_loss
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            counter = 0
            
        else:
        
            counter += 1
            
            print(f'EarlyStopping counter: {counter} out of {patience}')
            
        if counter >= patience:
            print("Early stopping triggered. Stopping training.")
            break
            
        sheduler.step()
        
    return model

In [12]:
class FullModel(nn.Module):
    def __init__(self, backbone, classifier):
        super().__init__()
        self.backbone = backbone
        self.classifier = classifier

    def forward(self, x):
        x = self.backbone(x)
        x = self.classifier(x)
        return x


model = FullModel(backbone, classifier).to(device)

In [13]:
criterion = nn.CrossEntropyLoss()
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(trainable_params, lr = 0.0001)
sheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-7)

In [14]:
model = train_model(model, criterion, optimizer, num_epochs=100, sheduler=sheduler)

Epochs:   0%|          | 0/100 [00:00<?, ?epoch/s]

  train 1/100:   0%|          | 0/234 [00:00<?, ?batch/s]

  val   1/100:   0%|          | 0/30 [00:00<?, ?batch/s]

Epoch 1 		 Training Loss: 0.5849043192007602 		 Validation Loss: 0.5675308135648568
Validation Loss Decreased(inf--->17.025924) 	 Saving The Model


  train 2/100:   0%|          | 0/234 [00:00<?, ?batch/s]

...

  val   40/100:   0%|          | 0/30 [00:00<?, ?batch/s]

Epoch 40 		 Training Loss: 0.46410804783177173 		 Validation Loss: 0.4509121845165888
EarlyStopping counter: 9 out of 10


  train 41/100:   0%|          | 0/234 [00:00<?, ?batch/s]

  val   41/100:   0%|          | 0/30 [00:00<?, ?batch/s]

Epoch 41 		 Training Loss: 0.4603665780562621 		 Validation Loss: 0.45727967595060665
EarlyStopping counter: 10 out of 10
Early stopping triggered. Stopping training.


In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [ ]:
BEST_MODEL_PATH = "/media/imagenesmedicas/DATA1/01-ImagenesMedicas-US1/13-PregradoJulian/Federal Learning/infraestructura federada/Federal-Learning/runs/exp74_resnet50_radimagenet_alllayerfreeze/best_model.pth"

model.load_state_dict(torch.load(BEST_MODEL_PATH))
model.to(device)


In [ ]:
# Curva de loss (training vs validation) por época.
# Usa train_loss_history / val_loss_history, poblados dentro de train_model()
# (celda 4a3328a4) — si el kernel se reinicia y esta celda se corre sola sin
# volver a entrenar, ambas listas estarán vacías.
import matplotlib.pyplot as plt

best_epoch = int(np.argmin(val_loss_history)) + 1  # 1-indexed, coincide con BEST_MODEL_PATH

for epoch, (t_loss, v_loss) in enumerate(zip(train_loss_history, val_loss_history), start=1):
    marker = "  <-- best (min val loss)" if epoch == best_epoch else ""
    print(f"Epoch {epoch:3d} \t Training Loss: {t_loss:.6f} \t Validation Loss: {v_loss:.6f}{marker}")

fig, ax = plt.subplots(figsize=(8, 5))
epochs_range = range(1, len(train_loss_history) + 1)
ax.plot(epochs_range, train_loss_history, label="Training loss", color="#1f77b4")
ax.plot(epochs_range, val_loss_history, label="Validation loss", color="#d62728")
ax.axvline(best_epoch, linestyle="--", color="gray", lw=1, label=f"Best epoch ({best_epoch})")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("exp74 — Training vs Validation loss")
ax.legend()
fig.tight_layout()

loss_curve_path = RUN_DIR / "plots" / "loss_curve.png"
loss_curve_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(loss_curve_path, dpi=150, bbox_inches="tight")
print(f"\nFigura guardada en: {loss_curve_path}")
plt.show()

In [ ]:
# train_model() solo guarda BEST_MODEL_PATH cuando la validation loss mejora
# (dentro del loop); el objeto `model` que devuelve, en cambio, queda tal
# cual terminó la última época (no se recarga el mejor checkpoint dentro de
# la función). Por eso esta celda debe ir aquí, justo después de
# `model = train_model(...)` y ANTES de la celda de evaluación en test
# (70d18898), que sí hace `model.load_state_dict(torch.load(BEST_MODEL_PATH))`
# y pisaría los pesos de la última época en memoria con los del mejor checkpoint.
LAST_MODEL_PATH = RUN_DIR / "last_model.pth"
torch.save(model.state_dict(), LAST_MODEL_PATH)
print(f"Modelo de la última época guardado en: {LAST_MODEL_PATH}")

In [19]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# Evaluate the best (lowest val-loss) checkpoint on the held-out test split.
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()

all_labels, all_preds, all_probs = [], [], []
with torch.no_grad():
    for data, labels in tqdm(test, desc="Test", unit="batch"):
        data = data.to(device, dtype=torch.float)
        logits = model(data)
        probs = torch.softmax(logits, dim=-1)[:, 1]  # P(class index 1)
        preds = logits.argmax(dim=-1)

        all_labels.extend(labels.numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

test_acc = accuracy_score(all_labels, all_preds)
test_auc = roc_auc_score(all_labels, all_probs)

idx_to_label = {v: k for k, v in train_processed.label_to_idx.items()}
print(f"label_to_idx: {train_processed.label_to_idx}  (positive class for AUC = {idx_to_label[1]!r})")
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test ROC AUC:  {test_auc:.4f}")
print()
print(classification_report(all_labels, all_preds, target_names=[idx_to_label[0], idx_to_label[1]]))

Test:   0%|          | 0/30 [00:00<?, ?batch/s]

KeyboardInterrupt: 